# 🏍️ Sistema Experto para un Taller Mecánico de Motos
**Inteligencia Artificial II – Proyecto final (Sistema basado en conocimiento con interfaz gráfica)**

Integrantes: Santiago Garzon  ·  Alvaro Zarate

El sistema recibe **síntomas** que el cliente describe o que el mecánico observa en la moto y, mediante **encadenamiento hacia adelante**, deriva **diagnósticos** y **recomendaciones** de reparación, explicando cada paso del razonamiento.

**Restricciones que se respetan:** Python puro, **sin importar ninguna librería** (ni siquiera `json`, `random`, etc.). La interfaz gráfica se construye con HTML/CSS/JS generado desde Python y se muestra con `display`, que ya viene incluido en el kernel de Jupyter/Colab.

| Parte |
|---|
| 1. Introducción y diseño |
| 2. Base de conocimiento (predicados y reglas) |
| 3. Motor de inferencia y explicación |
| 4. Validación (casos de prueba) |
| 5. Interfaz gráfica |
| 6. Limitaciones y conclusiones |

## 1. Introducción y diseño

**Dominio:** diagnóstico de fallas comunes en motocicletas de cilindraje bajo y medio (encendido, alimentación, motor, transmisión y frenos).

**Justificación:** en un taller, buena parte del tiempo se pierde haciendo siempre las mismas preguntas. Un sistema experto captura ese conocimiento y ayuda a decidir qué revisar y qué repuesto cambiar.

**Arquitectura**

```
 ┌────────────────┐   síntomas    ┌──────────────────────────┐   hechos + reglas   ┌─────────────────┐
 │ Interfaz (HTML)│ ────────────► │  Motor de inferencia     │ ──────────────────► │ Explicación     │
 │ checkboxes,    │               │  encadenamiento_adelante │                     │ porque(), reglas│
 │ casos, botones │ ◄──────────── │  + validación + ciclos   │ ◄── Base de         │ aplicadas       │
 └────────────────┘  resultados   └──────────────────────────┘     conocimiento    └─────────────────┘
```

**Flujo de datos:** el usuario marca síntomas → se validan → el motor dispara reglas hasta que no hay cambios → se muestran diagnósticos, recomendaciones, nivel de certeza y la cadena de razonamiento.

**Cómo se representan los hechos** (igual que en el notebook de ejemplo): cada hecho es un texto dentro de un `set`; una regla es un diccionario con `si` (lista de condiciones) y `entonces` (conclusión). Se añaden `descripcion` y `certeza` (factor entre 0 y 1) como extra.

## 2. Base de conocimiento

### 2.1 Predicados del dominio
Todos los predicados son **unarios** sobre una moto `m`:

- **Tipo:** `P(m)` con dominio `Moto` (la moto que está en el taller) y rango `{Verdadero, Falso}`.
- Se trabaja con **mundo cerrado**: lo que no se marca como observado se considera falso.
- Hay cuatro clases: **síntomas** (entrada), **conclusiones intermedias**, **diagnósticos** y **recomendaciones**.

| Clase | Predicados (notación lógica) |
|---|---|
| Síntomas (15) | `MarchaNoGira(m)`, `LucesDebiles(m)`, `MarchaGira(m)`, `SinChispa(m)`, `BujiaSucia(m)`, `NoLlegaGasolina(m)`, `Tirones(m)`, `PerdidaPotencia(m)`, `FiltroAireSucio(m)`, `HumoAzul(m)`, `ConsumoAceite(m)`, `RuidoCadena(m)`, `CadenaFloja(m)`, `FrenoChirrido(m)`, `RecorridoFrenoLargo(m)` |
| Intermedias (2) | `FallaEncendido(m)`, `MezclaIrregular(m)` |
| Diagnósticos (6) | `BateriaDescargada(m)`, `BujiaDanada(m)`, `FiltroAireObstruido(m)`, `DesgasteAros(m)`, `KitArrastreDesgastado(m)`, `PastillasDesgastadas(m)` |
| Recomendaciones (7) | `RecCargarBateria(m)`, `RecCambiarBujia(m)`, `RecCambiarFiltroAire(m)`, `RecRevisionMotor(m)`, `RecCambiarKitArrastre(m)`, `RecCambiarPastillas(m)`, `RecRevisarCombustible(m)` |

> Si el profesor exige estrictamente 8–12 predicados, se pueden agrupar las 15 observaciones en menos familias; aquí se mantienen separadas para que cada síntoma sea una casilla de la interfaz.

In [1]:
# Predicados: nombre -> (clase, grupo/sistema de la moto, texto legible)
PREDICADOS = {
    # --- Síntomas (los que el usuario puede marcar) ---
    "marcha_no_gira":     ("sintoma", "Encendido y electricidad", "La marcha no hace girar el motor (solo clic o silencio)"),
    "luces_debiles":      ("sintoma", "Encendido y electricidad", "Luces y pito se ven u oyen débiles"),
    "marcha_gira":        ("sintoma", "Encendido y electricidad", "La marcha gira el motor pero no enciende"),
    "sin_chispa":         ("sintoma", "Encendido y electricidad", "No hay chispa en la bujía"),
    "bujia_sucia":        ("sintoma", "Encendido y electricidad", "Bujía negra, húmeda o con carbón"),
    "no_llega_gasolina":  ("sintoma", "Alimentación y motor", "No llega gasolina al carburador o inyector"),
    "tirones":            ("sintoma", "Alimentación y motor", "La moto da tirones al acelerar"),
    "perdida_potencia":   ("sintoma", "Alimentación y motor", "Pérdida de potencia"),
    "filtro_aire_sucio":  ("sintoma", "Alimentación y motor", "Filtro de aire sucio u obstruido"),
    "humo_azul":          ("sintoma", "Alimentación y motor", "Humo azulado por el escape"),
    "consumo_aceite":     ("sintoma", "Alimentación y motor", "Consumo excesivo de aceite"),
    "ruido_cadena":       ("sintoma", "Transmisión", "Ruido metálico en la cadena"),
    "cadena_floja":       ("sintoma", "Transmisión", "Cadena floja o con holgura excesiva"),
    "freno_chirrido":     ("sintoma", "Frenos", "Chirrido al frenar"),
    "recorrido_freno_largo": ("sintoma", "Frenos", "La palanca o pedal de freno recorre demasiado"),
    # --- Conclusiones intermedias ---
    "falla_encendido":    ("intermedio", "Conclusión", "Falla en el sistema de encendido"),
    "mezcla_irregular":   ("intermedio", "Conclusión", "Mezcla aire-combustible irregular"),
    # --- Diagnósticos ---
    "bateria_descargada":      ("diagnostico", "Diagnóstico", "Batería descargada o en mal estado"),
    "bujia_danada":            ("diagnostico", "Diagnóstico", "Bujía sucia o dañada"),
    "filtro_aire_obstruido":   ("diagnostico", "Diagnóstico", "Filtro de aire obstruido"),
    "desgaste_aros":           ("diagnostico", "Diagnóstico", "Desgaste de aros o retenes del motor"),
    "kit_arrastre_desgastado": ("diagnostico", "Diagnóstico", "Kit de arrastre desgastado"),
    "pastillas_desgastadas":   ("diagnostico", "Diagnóstico", "Pastillas de freno desgastadas"),
    # --- Recomendaciones ---
    "recomendar_cargar_o_cambiar_bateria": ("recomendacion", "Recomendación", "Cargar la batería; si no retiene carga, cambiarla"),
    "recomendar_cambiar_bujia":            ("recomendacion", "Recomendación", "Cambiar la bujía y revisar su calibración"),
    "recomendar_cambiar_filtro_aire":      ("recomendacion", "Recomendación", "Limpiar o cambiar el filtro de aire"),
    "recomendar_revision_motor":           ("recomendacion", "Recomendación", "Llevar el motor a revisión (aros, guías y retenes)"),
    "recomendar_cambiar_kit_arrastre":     ("recomendacion", "Recomendación", "Cambiar cadena, piñón y corona (kit de arrastre)"),
    "recomendar_cambiar_pastillas":        ("recomendacion", "Recomendación", "Cambiar pastillas de freno y revisar los discos"),
    "recomendar_revisar_combustible":      ("recomendacion", "Recomendación", "Revisar llave de gasolina, filtro, bomba y carburador/inyectores"),
}

# Pares de síntomas que no pueden darse a la vez (validación de entrada)
CONTRADICCIONES = [("marcha_gira", "marcha_no_gira")]

def texto(hecho):
    """Texto legible de un hecho (si no está en PREDICADOS devuelve el mismo nombre)."""
    return PREDICADOS[hecho][2] if hecho in PREDICADOS else hecho

print("Predicados definidos:", len(PREDICADOS))

Predicados definidos: 30


### 2.2 Reglas en lógica de predicados (15 reglas)

| Regla | Fórmula |
|---|---|
| R1 | `∀m (MarchaGira(m) ∧ SinChispa(m) → FallaEncendido(m))` |
| R2 | `∀m (Tirones(m) ∧ PerdidaPotencia(m) → MezclaIrregular(m))` |
| R3 | `∀m (MarchaNoGira(m) ∧ LucesDebiles(m) → BateriaDescargada(m))` |
| R4 | `∀m (FallaEncendido(m) ∧ BujiaSucia(m) → BujiaDanada(m))` |
| R5 | `∀m (MezclaIrregular(m) ∧ FiltroAireSucio(m) → FiltroAireObstruido(m))` |
| R6 | `∀m (HumoAzul(m) ∧ ConsumoAceite(m) → DesgasteAros(m))` |
| R7 | `∀m (RuidoCadena(m) ∧ CadenaFloja(m) → KitArrastreDesgastado(m))` |
| R8 | `∀m (FrenoChirrido(m) ∧ RecorridoFrenoLargo(m) → PastillasDesgastadas(m))` |
| R9 | `∀m (MarchaGira(m) ∧ NoLlegaGasolina(m) → RecRevisarCombustible(m))` |
| R10 | `∀m (BateriaDescargada(m) → RecCargarBateria(m))` |
| R11 | `∀m (BujiaDanada(m) → RecCambiarBujia(m))` |
| R12 | `∀m (FiltroAireObstruido(m) → RecCambiarFiltroAire(m))` |
| R13 | `∀m (DesgasteAros(m) → RecRevisionMotor(m))` |
| R14 | `∀m (KitArrastreDesgastado(m) → RecCambiarKitArrastre(m))` |
| R15 | `∀m (PastillasDesgastadas(m) → RecCambiarPastillas(m))` |

**Justificación:** R1–R2 agrupan síntomas en causas probables (intermedias); R3–R8 confirman un diagnóstico exigiendo dos evidencias que se refuerzan; R9 es una recomendación directa porque "gira pero no enciende y no llega gasolina" ya señala el sistema de combustible; R10–R15 convierten cada diagnóstico en una acción concreta para el mecánico.

**Factor de certeza:** cada regla tiene un `certeza` entre 0 y 1. Los síntomas valen 1.0 y la certeza de una conclusión es `mínimo(certezas de las condiciones) × certeza de la regla`.

In [2]:
# Base de conocimiento (formato SI-ENTONCES, como en el notebook de ejemplo)
base_conocimiento = [
    # --- Conclusiones intermedias ---
    {"nombre": "R1", "descripcion": "Gira pero no hay chispa: falla de encendido",
     "si": ["marcha_gira", "sin_chispa"], "entonces": "falla_encendido", "certeza": 0.90},
    {"nombre": "R2", "descripcion": "Tirones con falta de potencia: mezcla irregular",
     "si": ["tirones", "perdida_potencia"], "entonces": "mezcla_irregular", "certeza": 0.85},

    # --- Diagnósticos ---
    {"nombre": "R3", "descripcion": "Sin giro de marcha y luces débiles: batería",
     "si": ["marcha_no_gira", "luces_debiles"], "entonces": "bateria_descargada", "certeza": 0.95},
    {"nombre": "R4", "descripcion": "Falla de encendido con bujía sucia: bujía dañada",
     "si": ["falla_encendido", "bujia_sucia"], "entonces": "bujia_danada", "certeza": 0.90},
    {"nombre": "R5", "descripcion": "Mezcla irregular con filtro sucio: filtro de aire",
     "si": ["mezcla_irregular", "filtro_aire_sucio"], "entonces": "filtro_aire_obstruido", "certeza": 0.90},
    {"nombre": "R6", "descripcion": "Humo azul y consumo de aceite: aros o retenes",
     "si": ["humo_azul", "consumo_aceite"], "entonces": "desgaste_aros", "certeza": 0.85},
    {"nombre": "R7", "descripcion": "Ruido y holgura en la cadena: kit de arrastre",
     "si": ["ruido_cadena", "cadena_floja"], "entonces": "kit_arrastre_desgastado", "certeza": 0.95},
    {"nombre": "R8", "descripcion": "Chirrido y recorrido largo: pastillas",
     "si": ["freno_chirrido", "recorrido_freno_largo"], "entonces": "pastillas_desgastadas", "certeza": 0.95},

    # --- Recomendación directa ---
    {"nombre": "R9", "descripcion": "Gira pero no llega gasolina: revisar combustible",
     "si": ["marcha_gira", "no_llega_gasolina"], "entonces": "recomendar_revisar_combustible", "certeza": 0.90},

    # --- Recomendaciones a partir de diagnósticos ---
    {"nombre": "R10", "descripcion": "Acción para batería descargada",
     "si": ["bateria_descargada"], "entonces": "recomendar_cargar_o_cambiar_bateria", "certeza": 0.95},
    {"nombre": "R11", "descripcion": "Acción para bujía dañada",
     "si": ["bujia_danada"], "entonces": "recomendar_cambiar_bujia", "certeza": 0.95},
    {"nombre": "R12", "descripcion": "Acción para filtro de aire obstruido",
     "si": ["filtro_aire_obstruido"], "entonces": "recomendar_cambiar_filtro_aire", "certeza": 0.95},
    {"nombre": "R13", "descripcion": "Acción para desgaste de aros",
     "si": ["desgaste_aros"], "entonces": "recomendar_revision_motor", "certeza": 0.90},
    {"nombre": "R14", "descripcion": "Acción para kit de arrastre desgastado",
     "si": ["kit_arrastre_desgastado"], "entonces": "recomendar_cambiar_kit_arrastre", "certeza": 0.95},
    {"nombre": "R15", "descripcion": "Acción para pastillas desgastadas",
     "si": ["pastillas_desgastadas"], "entonces": "recomendar_cambiar_pastillas", "certeza": 0.95},
]

print("Reglas en la base:", len(base_conocimiento))

Reglas en la base: 15


In [3]:
# Utilidades de reglas (mismo estilo del notebook de ejemplo)
def mostrar_regla(regla):
    condiciones = " Y ".join(regla["si"])
    print(f'{regla["nombre"]}: SI {condiciones} '
          f'ENTONCES {regla["entonces"]}  (certeza {regla["certeza"]})')

def se_cumple(regla, hechos):
    """Una regla se cumple cuando TODAS sus condiciones están en los hechos."""
    for condicion in regla["si"]:
        if condicion not in hechos:
            return False
    return True

for r in base_conocimiento:
    mostrar_regla(r)

R1: SI marcha_gira Y sin_chispa ENTONCES falla_encendido  (certeza 0.9)
R2: SI tirones Y perdida_potencia ENTONCES mezcla_irregular  (certeza 0.85)
R3: SI marcha_no_gira Y luces_debiles ENTONCES bateria_descargada  (certeza 0.95)
R4: SI falla_encendido Y bujia_sucia ENTONCES bujia_danada  (certeza 0.9)
R5: SI mezcla_irregular Y filtro_aire_sucio ENTONCES filtro_aire_obstruido  (certeza 0.9)
R6: SI humo_azul Y consumo_aceite ENTONCES desgaste_aros  (certeza 0.85)
R7: SI ruido_cadena Y cadena_floja ENTONCES kit_arrastre_desgastado  (certeza 0.95)
R8: SI freno_chirrido Y recorrido_freno_largo ENTONCES pastillas_desgastadas  (certeza 0.95)
R9: SI marcha_gira Y no_llega_gasolina ENTONCES recomendar_revisar_combustible  (certeza 0.9)
R10: SI bateria_descargada ENTONCES recomendar_cargar_o_cambiar_bateria  (certeza 0.95)
R11: SI bujia_danada ENTONCES recomendar_cambiar_bujia  (certeza 0.95)
R12: SI filtro_aire_obstruido ENTONCES recomendar_cambiar_filtro_aire  (certeza 0.95)
R13: SI desgaste_

## 3. Motor de inferencia

Se parte de la versión didáctica (`una_pasada`, una sola vuelta por las reglas) y luego se completa con el **encadenamiento hacia adelante** real, que repite pasadas hasta que no aparezcan hechos nuevos. Incluye:

- **Detección de ciclos infinitos:** cada pasada con cambios añade al menos un hecho nuevo y solo hay `len(base)` conclusiones posibles; si se supera `len(base)+1` pasadas, el motor lanza un error.
- **Logging** del proceso (lista de mensajes).
- **Manejo de "sin conclusión":** si no se deriva nada, el sistema lo informa y sugiere qué datos faltan (`que_falta`).

In [4]:
# Versión didáctica: una sola pasada
def una_pasada(base, sintomas):
    hechos = set(sintomas)  # copia
    for regla in base:
        if se_cumple(regla, hechos):
            hechos.add(regla["entonces"])
            print("Se disparó", regla["nombre"])
    return hechos

# Con el orden actual de la base, una pasada ya encadena R1 -> R4 -> R11
print(una_pasada(base_conocimiento, {"marcha_gira", "sin_chispa", "bujia_sucia"}))

Se disparó R1
Se disparó R4
Se disparó R11
{'bujia_danada', 'sin_chispa', 'falla_encendido', 'bujia_sucia', 'marcha_gira', 'recomendar_cambiar_bujia'}


In [5]:
# Validación de entradas
def validar_sintomas(sintomas):
    """Devuelve una lista de mensajes de error (vacía si todo está bien)."""
    errores = []
    if len(sintomas) == 0:
        errores.append("Seleccione al menos un síntoma.")
    for s in sintomas:
        if s not in PREDICADOS or PREDICADOS[s][0] != "sintoma":
            errores.append(f"'{s}' no es un síntoma conocido.")
    for a, b in CONTRADICCIONES:
        if a in sintomas and b in sintomas:
            errores.append(f"Síntomas contradictorios: '{texto(a)}' y '{texto(b)}'.")
    return errores

print(validar_sintomas({"marcha_gira", "marcha_no_gira"}))
print(validar_sintomas(set()))

["Síntomas contradictorios: 'La marcha gira el motor pero no enciende' y 'La marcha no hace girar el motor (solo clic o silencio)'."]
['Seleccione al menos un síntoma.']


In [6]:
# Encadenamiento hacia adelante
def encadenamiento_adelante(base, hechos_iniciales, registro=None):
    """
    Aplica encadenamiento hacia adelante.
    Returns: (hechos_finales, reglas_aplicadas)
      - reglas_aplicadas: lista (en orden de disparo) de copias de las reglas,
        cada una con la clave extra 'certeza_final'.
    registro: lista opcional donde se guardan los mensajes del log.
    """
    if registro is None:
        registro = []
    hechos = set(hechos_iniciales)
    certeza = {h: 1.0 for h in hechos}      # los síntomas tienen certeza 1
    reglas_aplicadas = []
    pasada = 0
    hubo_cambios = True
    while hubo_cambios:
        pasada += 1
        if pasada > len(base) + 1:          # protección contra ciclos infinitos
            raise RuntimeError("Posible ciclo infinito en la base de conocimiento")
        registro.append(f"--- Pasada {pasada} ---")
        hubo_cambios = False
        for regla in base:
            if se_cumple(regla, hechos) and regla["entonces"] not in hechos:
                c = min(certeza[x] for x in regla["si"]) * regla["certeza"]
                certeza[regla["entonces"]] = c
                hechos.add(regla["entonces"])
                aplicada = dict(regla)
                aplicada["certeza_final"] = c
                reglas_aplicadas.append(aplicada)
                registro.append(f'Se disparó {regla["nombre"]}: '
                                f'{" Y ".join(regla["si"])} => {regla["entonces"]} (certeza {c:.2f})')
                hubo_cambios = True
    if not reglas_aplicadas:
        registro.append("No se pudo derivar ninguna conclusión con los hechos dados.")
    return hechos, reglas_aplicadas

registro = []
hechos, aplicadas = encadenamiento_adelante(base_conocimiento,
                                            {"marcha_no_gira", "luces_debiles"}, registro)
print("\n".join(registro))
print(sorted(hechos))

--- Pasada 1 ---
Se disparó R3: marcha_no_gira Y luces_debiles => bateria_descargada (certeza 0.95)
Se disparó R10: bateria_descargada => recomendar_cargar_o_cambiar_bateria (certeza 0.90)
--- Pasada 2 ---
['bateria_descargada', 'luces_debiles', 'marcha_no_gira', 'recomendar_cargar_o_cambiar_bateria']


In [7]:
# Explicación: razonamiento completo, "¿por qué?" y "¿qué falta?"
def explicar_razonamiento(reglas_aplicadas):
    """Genera un texto legible con las reglas aplicadas en orden."""
    if not reglas_aplicadas:
        return "No se aplicó ninguna regla: con los síntomas dados no se puede concluir nada."
    lineas = []
    for i, r in enumerate(reglas_aplicadas, 1):
        lineas.append(f'Paso {i} · {r["nombre"]} ({r["descripcion"]}): '
                      f'SI {" Y ".join(texto(c) for c in r["si"])} '
                      f'ENTONCES {texto(r["entonces"])}  [certeza {r["certeza_final"]:.0%}]')
    return "\n".join(lineas)

def porque(conclusion, hechos_iniciales, reglas_aplicadas, nivel=0):
    """Explica por qué se llegó a una conclusión (cadena de razonamiento recursiva)."""
    origen = {r["entonces"]: r for r in reglas_aplicadas}
    sangria = "    " * nivel
    if conclusion in origen:
        regla = origen[conclusion]
        texto_out = f'{sangria}{conclusion} .- {regla["nombre"]} ({regla["descripcion"]})\n'
        for c in regla["si"]:
            texto_out += porque(c, hechos_iniciales, reglas_aplicadas, nivel + 1)
        return texto_out
    if conclusion in hechos_iniciales:
        return f"{sangria}{conclusion} .- síntoma ingresado\n"
    return f"{sangria}{conclusion} .- no se pudo derivar\n"

def que_falta(base, hechos):
    """Para cada regla no disparada con evidencia parcial, indica qué condiciones faltan."""
    faltantes = []
    for regla in base:
        if regla["entonces"] in hechos:
            continue
        presentes = [c for c in regla["si"] if c in hechos]
        ausentes = [c for c in regla["si"] if c not in hechos]
        if presentes and ausentes:
            faltantes.append((regla["nombre"], regla["entonces"], ausentes))
    return faltantes

# Demostración
h, ap = encadenamiento_adelante(base_conocimiento, {"marcha_gira", "sin_chispa", "bujia_sucia"})
print(explicar_razonamiento(ap))
print()
print(porque("recomendar_cambiar_bujia", {"marcha_gira", "sin_chispa", "bujia_sucia"}, ap))
print(que_falta(base_conocimiento, {"freno_chirrido"}))

Paso 1 · R1 (Gira pero no hay chispa: falla de encendido): SI La marcha gira el motor pero no enciende Y No hay chispa en la bujía ENTONCES Falla en el sistema de encendido  [certeza 90%]
Paso 2 · R4 (Falla de encendido con bujía sucia: bujía dañada): SI Falla en el sistema de encendido Y Bujía negra, húmeda o con carbón ENTONCES Bujía sucia o dañada  [certeza 81%]
Paso 3 · R11 (Acción para bujía dañada): SI Bujía sucia o dañada ENTONCES Cambiar la bujía y revisar su calibración  [certeza 77%]

recomendar_cambiar_bujia .- R11 (Acción para bujía dañada)
    bujia_danada .- R4 (Falla de encendido con bujía sucia: bujía dañada)
        falla_encendido .- R1 (Gira pero no hay chispa: falla de encendido)
            marcha_gira .- síntoma ingresado
            sin_chispa .- síntoma ingresado
        bujia_sucia .- síntoma ingresado

[('R8', 'pastillas_desgastadas', ['recorrido_freno_largo'])]


**Casos especiales manejados por el motor:**

| Situación | Comportamiento |
|---|---|
| Ningún síntoma / síntoma inexistente / síntomas contradictorios | `validar_sintomas` devuelve mensajes de error |
| Síntomas que no activan ninguna regla | El motor devuelve lista de reglas vacía y `explicar_razonamiento` lo indica |
| Evidencia parcial | `que_falta` sugiere los síntomas que completarían una regla |
| Una conclusión ya derivada por otra regla | No se vuelve a disparar (evita duplicados y ciclos) |
| Base mal diseñada con ciclos | `RuntimeError` por exceso de pasadas |

## 4. Validación: casos de prueba

| Caso | Descripción | Hechos iniciales | Reglas esperadas | Conclusión esperada |
|---|---|---|---|---|
| 1 | Moto que no da marcha (batería) | `marcha_no_gira`, `luces_debiles` | R3, R10 | `recomendar_cargar_o_cambiar_bateria` |
| 2 | Gira pero no enciende (bujía) | `marcha_gira`, `sin_chispa`, `bujia_sucia` | R1, R4, R11 | `recomendar_cambiar_bujia` |
| 3 | Tirones y falta de potencia | `tirones`, `perdida_potencia`, `filtro_aire_sucio` | R2, R5, R12 | `recomendar_cambiar_filtro_aire` |
| 4 | **Moto con mantenimiento vencido** (activa 13 de 15 reglas) | 13 síntomas (ver celda de datos de casos) | R1, R2, R4–R9, R11–R15 | 5 recomendaciones + combustible |
| 5 | Sin conclusión (síntoma aislado) | `luces_debiles` | ninguna | sin conclusión |
| 6 | Borde: evidencia parcial | `marcha_gira`, `sin_chispa` | R1 | solo `falla_encendido`, sin recomendación |
| 7 | Borde: entrada contradictoria | `marcha_gira`, `marcha_no_gira` | ninguna | error de validación |
| 8 | Borde: frenos incompletos | `freno_chirrido` | ninguna | sin conclusión; `que_falta` sugiere `recorrido_freno_largo` |

In [8]:
# Casos de prueba (datos)
casos = [
    {"caso": 1, "desc": "Batería",
     "sintomas": {"marcha_no_gira", "luces_debiles"},
     "reglas": {"R3", "R10"}, "debe_contener": {"recomendar_cargar_o_cambiar_bateria"}},
    {"caso": 2, "desc": "Bujía",
     "sintomas": {"marcha_gira", "sin_chispa", "bujia_sucia"},
     "reglas": {"R1", "R4", "R11"}, "debe_contener": {"recomendar_cambiar_bujia"}},
    {"caso": 3, "desc": "Filtro de aire",
     "sintomas": {"tirones", "perdida_potencia", "filtro_aire_sucio"},
     "reglas": {"R2", "R5", "R12"}, "debe_contener": {"recomendar_cambiar_filtro_aire"}},
    {"caso": 4, "desc": "Mantenimiento vencido (13 reglas)",
     "sintomas": {"marcha_gira", "sin_chispa", "bujia_sucia", "no_llega_gasolina",
                  "tirones", "perdida_potencia", "filtro_aire_sucio",
                  "humo_azul", "consumo_aceite", "ruido_cadena", "cadena_floja",
                  "freno_chirrido", "recorrido_freno_largo"},
     "reglas": {"R1", "R2", "R4", "R5", "R6", "R7", "R8", "R9", "R11", "R12", "R13", "R14", "R15"},
     "debe_contener": {"recomendar_cambiar_bujia", "recomendar_cambiar_filtro_aire",
                       "recomendar_revision_motor", "recomendar_cambiar_kit_arrastre",
                       "recomendar_cambiar_pastillas", "recomendar_revisar_combustible"}},
    {"caso": 5, "desc": "Sin conclusión",
     "sintomas": {"luces_debiles"}, "reglas": set(), "debe_contener": set()},
    {"caso": 6, "desc": "Evidencia parcial",
     "sintomas": {"marcha_gira", "sin_chispa"}, "reglas": {"R1"}, "debe_contener": {"falla_encendido"}},
    {"caso": 7, "desc": "Entrada contradictoria",
     "sintomas": {"marcha_gira", "marcha_no_gira"}, "reglas": "error", "debe_contener": set()},
    {"caso": 8, "desc": "Frenos incompletos",
     "sintomas": {"freno_chirrido"}, "reglas": set(), "debe_contener": set()},
]
print("Casos cargados:", len(casos))

Casos cargados: 8


In [9]:
# Ejecución automática de los casos y comparación con lo esperado
print(f'{"Caso":<5}{"Descripción":<36}{"Reglas obtenidas":<48}{"Estado"}')
print("-" * 100)
for c in casos:
    errores = validar_sintomas(c["sintomas"])
    if c["reglas"] == "error":
        ok = len(errores) > 0
        obtenidas = "error de validación"
    else:
        hechos, aplicadas = encadenamiento_adelante(base_conocimiento, c["sintomas"])
        nombres = {r["nombre"] for r in aplicadas}
        ok = (nombres == c["reglas"]) and c["debe_contener"].issubset(hechos) and not errores
        obtenidas = ", ".join(sorted(nombres, key=lambda n: int(n[1:]))) or "ninguna"
    print(f'{c["caso"]:<5}{c["desc"]:<36}{obtenidas:<48}{"✓" if ok else "✗"}')

Caso Descripción                         Reglas obtenidas                                Estado
----------------------------------------------------------------------------------------------------
1    Batería                             R3, R10                                         ✓
2    Bujía                               R1, R4, R11                                     ✓
3    Filtro de aire                      R2, R5, R12                                     ✓
4    Mantenimiento vencido (13 reglas)   R1, R2, R4, R5, R6, R7, R8, R9, R11, R12, R13, R14, R15✓
5    Sin conclusión                      ninguna                                         ✓
6    Evidencia parcial                   R1                                              ✓
7    Entrada contradictoria              error de validación                             ✓
8    Frenos incompletos                  ninguna                                         ✓


In [10]:
# Detalle del caso 4 (el más completo) y del caso de borde 8
hechos4, aplicadas4 = encadenamiento_adelante(base_conocimiento, casos[3]["sintomas"])
print(explicar_razonamiento(aplicadas4))
print("\nCaso 8 · ¿qué falta?")
for nombre, concl, faltan in que_falta(base_conocimiento, casos[7]["sintomas"]):
    print(f"  {nombre} ({concl}) necesitaría: {faltan}")

Paso 1 · R1 (Gira pero no hay chispa: falla de encendido): SI La marcha gira el motor pero no enciende Y No hay chispa en la bujía ENTONCES Falla en el sistema de encendido  [certeza 90%]
Paso 2 · R2 (Tirones con falta de potencia: mezcla irregular): SI La moto da tirones al acelerar Y Pérdida de potencia ENTONCES Mezcla aire-combustible irregular  [certeza 85%]
Paso 3 · R4 (Falla de encendido con bujía sucia: bujía dañada): SI Falla en el sistema de encendido Y Bujía negra, húmeda o con carbón ENTONCES Bujía sucia o dañada  [certeza 81%]
Paso 4 · R5 (Mezcla irregular con filtro sucio: filtro de aire): SI Mezcla aire-combustible irregular Y Filtro de aire sucio u obstruido ENTONCES Filtro de aire obstruido  [certeza 76%]
Paso 5 · R6 (Humo azul y consumo de aceite: aros o retenes): SI Humo azulado por el escape Y Consumo excesivo de aceite ENTONCES Desgaste de aros o retenes del motor  [certeza 85%]
Paso 6 · R7 (Ruido y holgura en la cadena: kit de arrastre): SI Ruido metálico en la cad

## 5. Interfaz gráfica

La interfaz se genera con una función de Python que:

1. Toma `PREDICADOS`, `base_conocimiento`, `CONTRADICCIONES` y `casos` y los inserta en una plantilla HTML (`repr` de Python produce literales válidos en JavaScript, así que **no hace falta ninguna librería**).
2. Incluye el mismo algoritmo de encadenamiento hacia adelante escrito en JavaScript, para que el navegador pueda razonar sin servidor. Es una traducción literal de la función `encadenamiento_adelante` de Python; la celda de ejecución de casos comprueba la versión Python y la interfaz usa la misma base de conocimiento.
3. Muestra el resultado con `display` (ya disponible en Jupyter/Colab sin `import`).

**Elementos de la interfaz:** casillas de síntomas agrupadas por sistema · botones de casos precargados · validación con mensajes claros · indicador de "analizando" · tarjetas de diagnóstico y recomendación con barra de certeza · botón **¿Por qué?** en cada conclusión · lista de reglas aplicadas en orden · registro de inferencia · aviso de "sin conclusión" con sugerencia de qué falta · botón **Limpiar**.

In [11]:
# Plantilla HTML/CSS/JS de la interfaz (texto; los datos se insertan en la celda 17)
PLANTILLA = r"""
<div id="__UID__" style="font-family:Segoe UI,Arial,sans-serif;background:#ffffff;color:#1f2937;border:1px solid #d1d5db;border-radius:12px;padding:16px;max-width:1100px">
<style>
#__UID__ *{box-sizing:border-box}
#__UID__ h2{margin:0 0 4px 0;font-size:22px}
#__UID__ h3{margin:14px 0 8px 0;font-size:16px;border-bottom:2px solid #e5e7eb;padding-bottom:4px}
#__UID__ .sub{color:#6b7280;font-size:13px;margin-bottom:10px}
#__UID__ .grid{display:grid;grid-template-columns:1fr 1.2fr;gap:18px}
#__UID__ .grupo{margin-bottom:8px;background:#f9fafb;border:1px solid #e5e7eb;border-radius:8px;padding:8px 10px}
#__UID__ .grupo b{font-size:13px;color:#b45309}
#__UID__ label{display:block;font-size:14px;padding:3px 0;cursor:pointer}
#__UID__ button{font-size:14px;border:0;border-radius:6px;padding:8px 14px;cursor:pointer;margin:3px 4px 3px 0}
#__UID__ .prim{background:#b45309;color:#fff}
#__UID__ .sec{background:#e5e7eb;color:#1f2937}
#__UID__ .caso{background:#fef3c7;color:#92400e;font-size:12px;padding:5px 9px}
#__UID__ button:disabled{opacity:.5;cursor:wait}
#__UID__ .msg{padding:8px 10px;border-radius:6px;font-size:14px;margin-bottom:8px;display:none}
#__UID__ .msg.ok{background:#dcfce7;color:#166534;display:block}
#__UID__ .msg.err{background:#fee2e2;color:#991b1b;display:block}
#__UID__ .msg.info{background:#dbeafe;color:#1e40af;display:block}
#__UID__ .msg.warn{background:#fef9c3;color:#854d0e;display:block}
#__UID__ .card{border-radius:8px;padding:10px 12px;margin:6px 0;border-left:6px solid}
#__UID__ .diag{background:#fff7ed;border-color:#ea580c}
#__UID__ .rec{background:#f0fdf4;border-color:#16a34a}
#__UID__ .card .t{font-weight:600;font-size:15px}
#__UID__ .barra{height:8px;background:#e5e7eb;border-radius:4px;margin-top:6px;overflow:hidden}
#__UID__ .barra div{height:100%;background:#16a34a}
#__UID__ .cert{font-size:12px;color:#4b5563}
#__UID__ .por{display:none;background:#fff;border:1px dashed #9ca3af;border-radius:6px;padding:6px 10px;margin-top:8px;font-size:13px}
#__UID__ .por ul{margin:2px 0 2px 16px;padding-left:12px}
#__UID__ .tag{background:#fde68a;border-radius:4px;padding:0 5px;font-size:12px;font-weight:600}
#__UID__ ol{font-size:13px;padding-left:20px}
#__UID__ ol li{margin-bottom:4px}
#__UID__ pre{background:#111827;color:#d1fae5;font-size:12px;padding:8px;border-radius:6px;overflow:auto;max-height:180px;margin:0}
#__UID__ details summary{cursor:pointer;font-size:13px;color:#4b5563}
@media (max-width:760px){#__UID__ .grid{grid-template-columns:1fr}}
</style>

<h2>🏍️ Taller Experto de Motos</h2>
<div class="sub">Marque los síntomas observados y pulse <b>Diagnosticar</b>. El sistema razona con encadenamiento hacia adelante.</div>

<div class="grid">
  <div>
    <h3>1. Síntomas</h3>
    <div class="casos"></div>
    <div class="lista"></div>
    <button class="prim b-diag">🔧 Diagnosticar</button>
    <button class="sec b-limpiar">🧹 Limpiar</button>
  </div>
  <div>
    <h3>2. Resultados</h3>
    <div class="msg"></div>
    <div class="res"></div>
    <h3>3. Explicación del razonamiento</h3>
    <div class="exp"><span class="cert">Aún no se ha ejecutado un diagnóstico.</span></div>
  </div>
</div>

<script>
(function(){
  var BASE = __BASE__;
  var PRED = __PRED__;
  var CONTRA = __CONTRA__;
  var CASOS = __CASOS__;
  var root = document.getElementById("__UID__");
  function $(s){ return root.querySelector(s); }
  function lab(h){ return PRED[h] ? PRED[h][2] : h; }

  /* ---------- Motor: traducción de encadenamiento_adelante ---------- */
  function motor(sint){
    var hechos = {}, cert = {}, origen = {}, aplicadas = [], reg = [];
    sint.forEach(function(s){ hechos[s] = true; cert[s] = 1; });
    var cambios = true, pasada = 0;
    while(cambios){
      pasada++;
      if(pasada > BASE.length + 1) throw new Error("Posible ciclo infinito en la base de conocimiento");
      reg.push("--- Pasada " + pasada + " ---");
      cambios = false;
      BASE.forEach(function(r){
        var cumple = r.si.every(function(c){ return hechos[c]; });
        if(cumple && !hechos[r.entonces]){
          var c = Math.min.apply(null, r.si.map(function(x){ return cert[x]; })) * r.certeza;
          hechos[r.entonces] = true; cert[r.entonces] = c; origen[r.entonces] = r; aplicadas.push(r);
          reg.push("Se disparó " + r.nombre + ": " + r.si.join(" Y ") + " => " + r.entonces + " (certeza " + c.toFixed(2) + ")");
          cambios = true;
        }
      });
    }
    if(aplicadas.length === 0) reg.push("No se pudo derivar ninguna conclusión con los hechos dados.");
    return {hechos:hechos, cert:cert, origen:origen, aplicadas:aplicadas, reg:reg};
  }

  function validar(sint){
    var e = [];
    if(sint.length === 0) e.push("Seleccione al menos un síntoma.");
    CONTRA.forEach(function(p){
      if(sint.indexOf(p[0]) >= 0 && sint.indexOf(p[1]) >= 0)
        e.push("Síntomas contradictorios: «" + lab(p[0]) + "» y «" + lab(p[1]) + "».");
    });
    return e;
  }

  function queFalta(res){
    var out = [];
    BASE.forEach(function(r){
      if(res.hechos[r.entonces]) return;
      var pres = r.si.filter(function(c){ return res.hechos[c]; });
      var aus = r.si.filter(function(c){ return !res.hechos[c]; });
      if(pres.length && aus.length) out.push({r:r, aus:aus});
    });
    return out;
  }

  function porque(h, res){
    var r = res.origen[h];
    if(!r) return "<li><b>" + lab(h) + "</b> — síntoma ingresado</li>";
    return "<li><b>" + lab(h) + "</b> ← <span class='tag'>" + r.nombre + "</span> " + r.descripcion +
           "<ul>" + r.si.map(function(c){ return porque(c, res); }).join("") + "</ul></li>";
  }

  /* ---------- Interfaz ---------- */
  var grupos = {};
  Object.keys(PRED).forEach(function(k){
    if(PRED[k][0] !== "sintoma") return;
    (grupos[PRED[k][1]] = grupos[PRED[k][1]] || []).push(k);
  });
  var html = "";
  Object.keys(grupos).forEach(function(g){
    html += "<div class='grupo'><b>" + g + "</b>";
    grupos[g].forEach(function(k){
      html += "<label><input type='checkbox' value='" + k + "'> " + lab(k) + "</label>";
    });
    html += "</div>";
  });
  $(".lista").innerHTML = html;

  var chtml = "<div class='sub'>Casos precargados:</div>";
  CASOS.forEach(function(c, i){
    chtml += "<button class='caso' data-i='" + i + "'>Caso " + c[0] + " · " + c[1] + "</button>";
  });
  $(".casos").innerHTML = chtml;

  function msg(t, tipo){ var m = $(".msg"); m.className = "msg " + tipo; m.textContent = t; }
  function marcados(){
    return Array.prototype.map.call(root.querySelectorAll(".lista input:checked"), function(i){ return i.value; });
  }
  function limpiar(){
    Array.prototype.forEach.call(root.querySelectorAll(".lista input"), function(i){ i.checked = false; });
    $(".res").innerHTML = ""; $(".msg").className = "msg";
    $(".exp").innerHTML = "<span class='cert'>Aún no se ha ejecutado un diagnóstico.</span>";
  }

  function render(sint, res){
    var diag = [], rec = [];
    res.aplicadas.forEach(function(r){
      var t = PRED[r.entonces] ? PRED[r.entonces][0] : "";
      if(t === "diagnostico") diag.push(r.entonces);
      if(t === "recomendacion") rec.push(r.entonces);
    });
    var out = "";
    function tarjeta(h, cls, icono){
      var p = Math.round(res.cert[h] * 100);
      return "<div class='card " + cls + "'><div class='t'>" + icono + " " + lab(h) + "</div>" +
        "<div class='cert'>Certeza: " + p + "%</div><div class='barra'><div style='width:" + p + "%'></div></div>" +
        "<button class='sec bpor' data-h='" + h + "'>¿Por qué?</button>" +
        "<div class='por'><ul>" + porque(h, res) + "</ul></div></div>";
    }
    if(diag.length){ out += "<b>Diagnósticos</b>"; diag.forEach(function(h){ out += tarjeta(h, "diag", "🩺"); }); }
    if(rec.length){ out += "<br><b>Recomendaciones</b>"; rec.forEach(function(h){ out += tarjeta(h, "rec", "✅"); }); }

    var falta = queFalta(res);
    if(res.aplicadas.length === 0){
      out = "<div class='card diag'><div class='t'>⚠️ Sin conclusión</div>" +
            "<div class='cert'>Con los síntomas marcados no se puede derivar ningún diagnóstico.</div></div>";
    } else if(!rec.length){
      out += "<div class='cert'>Se obtuvo una conclusión parcial pero todavía ninguna recomendación.</div>";
    }
    if(falta.length){
      out += "<br><b>Para llegar más lejos, verifique:</b><ul style='font-size:13px'>";
      falta.forEach(function(f){
        out += "<li>" + lab(f.r.entonces) + " → falta: " + f.aus.map(lab).join(", ") + "</li>";
      });
      out += "</ul>";
    }
    $(".res").innerHTML = out;

    var e = "";
    if(res.aplicadas.length){
      e += "<ol>";
      res.aplicadas.forEach(function(r){
        e += "<li><span class='tag'>" + r.nombre + "</span> SI " + r.si.map(lab).join(" <b>Y</b> ") +
             " <b>ENTONCES</b> " + lab(r.entonces) + " <span class='cert'>(certeza " + Math.round(res.cert[r.entonces]*100) + "%)</span></li>";
      });
      e += "</ol>";
    } else {
      e += "<span class='cert'>No se aplicó ninguna regla.</span>";
    }
    e += "<details><summary>Ver registro de inferencia</summary><pre>" + res.reg.join("\n") + "</pre></details>";
    $(".exp").innerHTML = e;
  }

  function diagnosticar(){
    var sint = marcados();
    var errs = validar(sint);
    if(errs.length){ msg(errs.join(" "), "err"); $(".res").innerHTML = ""; return; }
    var b = $(".b-diag");
    b.disabled = true; msg("Analizando síntomas…", "info");
    setTimeout(function(){
      try{
        var res = motor(sint);
        render(sint, res);
        if(res.aplicadas.length) msg("Diagnóstico completado: " + res.aplicadas.length + " regla(s) aplicada(s).", "ok");
        else msg("No se pudo concluir nada con estos síntomas.", "warn");
      }catch(err){ msg("Error interno: " + err.message, "err"); }
      b.disabled = false;
    }, 350);
  }

  $(".b-diag").onclick = diagnosticar;
  $(".b-limpiar").onclick = limpiar;
  root.addEventListener("click", function(ev){
    var t = ev.target;
    if(t.classList.contains("bpor")){
      var caja = t.nextElementSibling;
      var visible = caja.style.display === "block";
      caja.style.display = visible ? "none" : "block";
      t.textContent = visible ? "¿Por qué?" : "Ocultar";
    }
    if(t.classList.contains("caso")){
      limpiar();
      var c = CASOS[parseInt(t.getAttribute("data-i"), 10)];
      c[2].forEach(function(s){
        var el = root.querySelector(".lista input[value='" + s + "']");
        if(el) el.checked = true;
      });
      diagnosticar();
    }
  });
})();
</script>
</div>
"""

In [12]:
# Construcción y despliegue de la interfaz
def construir_interfaz():
    """Inserta la base de conocimiento en la plantilla y devuelve el HTML final."""
    uid = "taller" + str(id([]))                       # identificador único por ejecución
    pred_js = {k: list(v) for k, v in PREDICADOS.items()}
    casos_js = [[c["caso"], c["desc"], sorted(c["sintomas"])]
                for c in casos if c["reglas"] != "error"]   # el caso 7 es de validación, no se precarga
    pagina = (PLANTILLA.replace("__UID__", uid)
                       .replace("__BASE__", repr(base_conocimiento))
                       .replace("__PRED__", repr(pred_js))
                       .replace("__CONTRA__", repr([list(p) for p in CONTRADICCIONES]))
                       .replace("__CASOS__", repr(casos_js)))
    return pagina

HTML_FINAL = construir_interfaz()

# 1) Mostrar dentro del notebook (display ya existe en el kernel, sin import)
display({"text/html": HTML_FINAL}, raw=True)

# 2) Guardar una copia independiente que se abre con doble clic en cualquier navegador
archivo = open("interfaz_taller_motos.html", "w", encoding="utf-8")
archivo.write("<!DOCTYPE html><html><head><meta charset='utf-8'><title>Taller Experto de Motos</title></head>"
              "<body style='background:#f3f4f6;padding:20px'>" + HTML_FINAL + "</body></html>")
archivo.close()
print("Copia guardada como interfaz_taller_motos.html")

🏍️ Taller Experto de Motos 
 Marque los síntomas observados y pulse Diagnosticar . El sistema razona con encadenamiento hacia adelante. 

 
 
 1. Síntomas 
 
 
 🔧 Diagnosticar 
 🧹 Limpiar 
 
 
 2. Resultados 
 
 
 3. Explicación del razonamiento 
 Aún no se ha ejecutado un diagnóstico.

Copia guardada como interfaz_taller_motos.html


## 6. Limitaciones, conclusiones y mejoras

**Limitaciones**
- Mundo cerrado: un síntoma no marcado se asume ausente; el sistema no pregunta.
- Las reglas cubren fallas comunes, no diagnósticos de inyección electrónica avanzada ni fallas simultáneas muy raras.
- Los factores de certeza son estimaciones del equipo, no estadísticas reales del taller.
- Una conclusión se deriva una sola vez; si varias reglas la apoyan, no se combinan sus certezas.

**Logros:** base de 15 reglas con formalización lógica, motor con encadenamiento hacia adelante, detección de ciclos, explicación con `porque()` y `que_falta()`, 8 casos validados e interfaz gráfica sin librerías.

**Mejoras futuras:** preguntas guiadas (encadenamiento hacia atrás), combinación de certezas, historial de motos del cliente, cotización de repuestos asociada a cada recomendación.

**Cómo se integra la interfaz:** los datos viajan de Python a JavaScript dentro de la plantilla (celda de construcción de la interfaz); la lógica de inferencia de ambos lados es la misma, de modo que lo validado en la ejecución automática de casos es lo que muestra la pantalla.